In [1]:
import torch
import datasetMaker, modelMaker, training
import torchvision.transforms as transforms
from pathlib import Path
from torch.utils.data import DataLoader

In [2]:
sample_path = Path.cwd().parent / "ChicagoFSWild-Cropped"
csv_name = Path.cwd().parent / "ChicagoFSWild.csv"

In [3]:
tokenizer = datasetMaker.ASLVulcanTokenizer()

In [4]:
train_dataset = datasetMaker.ChicagoFSWildDataset(csv_file=csv_name, root_dir=sample_path, tokenizer=tokenizer)
print(f"Train Dataset size: {len(train_dataset)}")
    
if len(train_dataset) > 0:
    frames, tag = train_dataset[0]
    print(f"Frames shape: {frames.shape}")
    print(f"Tag (encoded label): {tag}")

Train Dataset size: 5455
Frames shape: torch.Size([16, 3, 224, 224])
Tag (encoded label): tensor([4, 3])


In [5]:
test_dataset = datasetMaker.ChicagoFSWildDataset(csv_file=csv_name, root_dir=sample_path, tokenizer=tokenizer, partition='test')
print(f"Test Dataset size: {len(test_dataset)}")
    
if len(test_dataset) > 0:
    frames, tag = test_dataset[0]
    print(f"Frames shape: {frames.shape}")
    print(f"Tag (encoded label): {tag}")

Test Dataset size: 868
Frames shape: torch.Size([9, 3, 224, 224])
Tag (encoded label): tensor([25, 12, 12])


In [6]:
val_dataset = datasetMaker.ChicagoFSWildDataset(csv_file=csv_name, root_dir=sample_path, tokenizer=tokenizer, partition='dev')
print(f"Validation Dataset size: {len(val_dataset)}")
    
if len(val_dataset) > 0:
    frames, tag = val_dataset[0]
    print(f"Frames shape: {frames.shape}")
    print(f"Tag (encoded label): {tag}")

Validation Dataset size: 981
Frames shape: torch.Size([66, 3, 224, 224])
Tag (encoded label): tensor([10, 15, 25, 13,  1,  9, 19,  5, 12])


In [7]:
train_dataloader = DataLoader(
        train_dataset, 
        batch_size=4, 
        shuffle=True, 
        collate_fn=datasetMaker.pad_collate
    )

In [8]:
test_dataloader = DataLoader(
        test_dataset, 
        batch_size=4, 
        shuffle=True, 
        collate_fn=datasetMaker.pad_collate
    )

In [9]:
val_dataloader = DataLoader(
        val_dataset, 
        batch_size=4, 
        shuffle=True, 
        collate_fn=datasetMaker.pad_collate
    )

In [10]:
from modelMaker import ResNetBiGRU,  ResNetBiLSTM

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [12]:
gru_model = ResNetBiGRU(num_classes=27, hidden_size=256, num_layers=2, dropout=0.5)
lstm_model = ResNetBiLSTM(num_classes=27, hidden_size=256, num_layers=2, dropout=0.5)

In [13]:
print(gru_model)

ResNetBiGRU(
  (backbone): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, 

In [14]:
print(lstm_model)

ResNetBiLSTM(
  (backbone): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1,

In [15]:
gru_trainable = sum(
    p.numel() for p in gru_model.parameters()
    if p.requires_grad
)

non_trainable = sum(
    p.numel() for p in gru_model.parameters()
    if not p.requires_grad
)

print(f"Trainable parameters (BiGRU):     {gru_trainable:,}")
print(f"Non-trainable parameters: {non_trainable:,}")
print(f"Total parameters:         {gru_trainable + non_trainable:,}")

trainable = sum(
    p.numel() for p in lstm_model.parameters()
    if p.requires_grad
)

lstm_non_trainable = sum(
    p.numel() for p in lstm_model.parameters()
    if not p.requires_grad
)

print(f"\n\nTrainable parameters (BiLSTM):     {trainable:,}")
print(f"Non-trainable parameters: {lstm_non_trainable:,}")
print(f"Total parameters:         {trainable + lstm_non_trainable:,}")

Trainable parameters (BiGRU):     12,611,355
Non-trainable parameters: 683,072
Total parameters:         13,294,427


Trainable parameters (BiLSTM):     13,268,763
Non-trainable parameters: 683,072
Total parameters:         13,951,835


In [16]:
from training import train_and_validate

In [18]:
train_and_validate(
    model=gru_model,
    train_loader=train_dataloader,
    val_loader=val_dataloader,
    tokenizer=tokenizer,
    max_epochs=1000,
    device=device,
    save_path=Path.cwd().parent/ "modelWeights" / "gru_best_model.pth"
)

Starting training on cuda...


Epoch 1 [Train]:   0%|          | 0/1364 [00:00<?, ?it/s]

Epoch 1 [Val]: 100%|██████████| 246/246 [02:20<00:00,  1.75it/s, loss=6.0784, cer=0.8000]



Epoch 1 Summary | Train Loss: 2.1709 | Val Loss: 2.5704 | Val CER: 0.7163 | LR: 0.001000
  -> Best model saved!


Epoch 2 [Val]: 100%|██████████| 246/246 [02:25<00:00,  1.69it/s, loss=1.8227, cer=1.0000]


...


Epoch 47 Summary | Train Loss: 0.6058 | Val Loss: 2.4930 | Val CER: 0.5220 | LR: 0.000125
  -> No improvement for 8 epoch(s).


Epoch 48 [Val]: 100%|██████████| 246/246 [02:28<00:00,  1.66it/s, loss=1.5977, cer=0.3000]



Epoch 48 Summary | Train Loss: 0.5865 | Val Loss: 2.4648 | Val CER: 0.5206 | LR: 0.000063
  -> No improvement for 9 epoch(s).


Epoch 49 [Val]: 100%|██████████| 246/246 [02:33<00:00,  1.60it/s, loss=2.0296, cer=0.3333]


Epoch 49 Summary | Train Loss: 0.5930 | Val Loss: 2.4443 | Val CER: 0.5222 | LR: 0.000063
  -> No improvement for 10 epoch(s).

[!] Validation CER plateaued for 10 epochs. Stopping training early.


In [19]:
train_and_validate(
    model=lstm_model,
    train_loader=train_dataloader,
    val_loader=val_dataloader,
    tokenizer=tokenizer,
    max_epochs=1000,
    device=device,
    save_path=Path.cwd().parent/ "modelWeights" / "lstm_best_model.pth"
)

Starting training on cuda...


Epoch 1 [Val]: 100%|██████████| 246/246 [02:50<00:00,  1.45it/s, loss=3.5023, cer=0.8333]



Epoch 1 Summary | Train Loss: 3.4726 | Val Loss: 3.3382 | Val CER: 0.8851 | LR: 0.001000
  -> Best model saved!


Epoch 2 [Val]: 100%|██████████| 246/246 [02:48<00:00,  1.46it/s, loss=3.4784, cer=1.0000]



Epoch 2 Summary | Train Loss: 3.3282 | Val Loss: 3.6186 | Val CER: 0.9057 | LR: 0.001000
  -> No improvement for 1 epoch(s).


...


Epoch 12 Summary | Train Loss: 3.1950 | Val Loss: 3.5084 | Val CER: 0.9314 | LR: 0.000500
  -> No improvement for 8 epoch(s).


Epoch 13 [Val]: 100%|██████████| 246/246 [02:56<00:00,  1.39it/s, loss=2.8098, cer=0.7500]



Epoch 13 Summary | Train Loss: 3.1870 | Val Loss: 3.3566 | Val CER: 0.9207 | LR: 0.000250
  -> No improvement for 9 epoch(s).


Epoch 14 [Val]: 100%|██████████| 246/246 [02:52<00:00,  1.43it/s, loss=3.4389, cer=1.0000]


Epoch 14 Summary | Train Loss: 3.1740 | Val Loss: 3.3407 | Val CER: 0.9017 | LR: 0.000250
  -> No improvement for 10 epoch(s).

[!] Validation CER plateaued for 10 epochs. Stopping training early.
